[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/bluesky.ipynb)

# Bluesky API

Collect public data from Bluesky with the `atproto` Python SDK: search posts by keyword and time, look up an account's profile, and list its followers. The three sections share one login and are meant to be run in order.

**Setup.** Install `atproto` and `python-dotenv`, then put your handle and an
[app password](https://bsky.app/settings/app-passwords) in a `.env` file next
to the notebook:

```
bluesky_username=your-handle.bsky.social
bluesky_password=your-app-password
```

Never commit the `.env` file. In Google Colab there is no `.env` file, so set
the two values with `os.environ["bluesky_username"] = "..."` in a cell you
delete before sharing, or use Colab's Secrets panel.

Log in once per session, not once per request: logins are limited to 30 per
5 minutes and 300 per day. All endpoints together are limited to 3,000
requests per 5 minutes per IP address.

Reference: [Bluesky HTTP API](https://endpoints.bsky.app/#bluesky-app/description/introduction).

## Log in

In [ ]:
from atproto import Client
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
client = Client()
client.login(os.getenv("bluesky_username"), os.getenv("bluesky_password"));

## Search posts

Search public posts by keyword with `app.bsky.feed.search_posts`, bound the search by time, and page through the results with the cursor.

In [ ]:
now = datetime.now(timezone.utc)
one_day_ago = now - timedelta(days=1)
iso_time_str = one_day_ago.isoformat().replace("+00:00", "Z")

In [ ]:
iso_time_str

The API wants a UTC timestamp that ends in `Z`. Output:

```python
'2026-08-29T04:53:36.621187Z'
```

In [ ]:
params = {
    "q": "cat",
    "limit": 10,
    "since": iso_time_str
}

In [ ]:
resps = client.app.bsky.feed.search_posts(
    params=params
)

In [ ]:
resps_dict = resps.model_dump()

In [ ]:
posts = resps_dict['posts']

In [ ]:
posts[0]

One post, trimmed. `record` is what the author wrote; everything around it is what the server adds (author profile, engagement counts, resolved media URLs).

```python
{'uri': 'at://did:plc:xxxxxxxxxxxxxxxxxxxxxxxx/app.bsky.feed.post/3lxggo53tpk2q',
 'cid': 'bafyreiho5rlyujjdo73lnq73aopbdlzir532eeamv3dqhadjxywy4klbhu',
 'author': {'did': 'did:plc:xxxxxxxxxxxxxxxxxxxxxxxx',
  'handle': 'example.bsky.social',
  'display_name': 'Example User',
  'created_at': '2023-12-25T05:38:35.649Z',
  'avatar': 'https://cdn.bsky.app/img/avatar/plain/...',
  'labels': [],
  'viewer': {...},          # your relationship to this account
  'py_type': 'app.bsky.actor.defs#profileViewBasic'},
 'record': {'text': 'imagine waking up to your cat like this',
  'created_at': '2026-08-29T01:52:25.598Z',
  'langs': ['en'],
  'reply': None,            # on a reply: {'parent': {...}, 'root': {...}}
  'facets': None,           # links, mentions, and tags inside the text
  'embed': {'images': [...], 'py_type': 'app.bsky.embed.images'},
  'py_type': 'app.bsky.feed.post'},
 'embed': {'images': [{'alt': '',
    'thumb': 'https://cdn.bsky.app/img/feed_thumbnail/plain/...',
    'fullsize': 'https://cdn.bsky.app/img/feed_fullsize/plain/...',
    'aspect_ratio': {'height': 2000, 'width': 1500, 'py_type': '...'},
    'py_type': 'app.bsky.embed.images#viewImage'}],
  'py_type': 'app.bsky.embed.images#view'},
 'indexed_at': '2026-08-29T01:52:36.068Z',
 'like_count': 0,
 'quote_count': 0,
 'reply_count': 0,
 'repost_count': 0,
 'labels': [],
 'threadgate': None,
 'viewer': {...},
 'py_type': 'app.bsky.feed.defs#postView'}
```

`indexed_at` is when the index saw the post; `record['created_at']` is when it was written. Use the second one for the posting time.

In [ ]:
resps_dict['cursor']

For `search_posts` the cursor is an offset. It is `'10'` after the first page of 10 posts and `'20'` after the second. When there is nothing more to read, the response has no cursor.

In [ ]:
params['cursor'] = resps_dict['cursor']

In [ ]:
resps_1 = client.app.bsky.feed.search_posts(
    params=params,
)

In [ ]:
resps_dict_1 = resps_1.model_dump()

In [ ]:
resps_dict_1['cursor']

## User profile

Look up one account's public profile with `app.bsky.actor.get_profile`: display name, description, follower and post counts, and the DID behind the handle.

In [ ]:
params = {
    "actor": "yang3kc.bsky.social"
}

In [ ]:
resps = client.app.bsky.actor.get_profile(
    params=params
)

In [ ]:
resps.model_dump()

Output, trimmed. The counts and the `created_at` field are the first things a bot detector looks at.

```python
{'did': 'did:plc:jqagnoo2ijf75flohmoqbrq4',
 'handle': 'yang3kc.bsky.social',
 'display_name': 'Kevin Yang',
 'description': 'Kaicheng Yang, PhD | Assistant professor @Binghamton University CS | ...',
 'created_at': '2023-08-16T15:06:26.185Z',
 'indexed_at': '2025-08-15T03:46:08.264Z',
 'followers_count': 693,
 'follows_count': 397,
 'posts_count': 93,
 'avatar': 'https://cdn.bsky.app/img/avatar/plain/...',
 'banner': None,
 'pinned_post': None,
 'labels': [],
 'associated': {'feedgens': 0, 'lists': 0, 'starter_packs': 0, 'labeler': False, ...},
 'viewer': {'blocked_by': False,
  'following': None,        # a follow URI if you follow this account
  'followed_by': None,      # a follow URI if it follows you
  'known_followers': {'count': 148, 'followers': [...]},   # followers you also follow
  ...},
 'py_type': 'app.bsky.actor.defs#profileViewDetailed'}
```

## Followers

List the accounts that follow one account with `app.bsky.graph.get_followers`. The response is paginated the same way as post search: pass the cursor back to get the next page. `get_follows` returns the accounts the actor follows, with the same shape.

In [ ]:
params = {
    "actor": "yang3kc.bsky.social"
}

In [ ]:
resps = client.app.bsky.graph.get_followers(
    params=params
)

In [ ]:
resps.model_dump()

Output, trimmed. `followers` is a list of profiles, 50 per page by default (`limit` goes up to 100), and `cursor` continues the list, as with search.

```python
{'subject': {'did': 'did:plc:jqagnoo2ijf75flohmoqbrq4',
  'handle': 'yang3kc.bsky.social',
  ...},
 'followers': [{'did': 'did:plc:xxxxxxxxxxxxxxxxxxxxxxxx',
   'handle': 'example-one.bsky.social',
   'display_name': 'Example One',
   'description': 'Short bio written by the account',
   'created_at': '2024-11-21T06:22:30.433Z',
   'indexed_at': '2025-08-24T03:54:15.864Z',
   'avatar': 'https://cdn.bsky.app/img/avatar/plain/...',
   'labels': [],
   'viewer': {...},
   'py_type': 'app.bsky.actor.defs#profileView'},
  {'did': 'did:plc:yyyyyyyyyyyyyyyyyyyyyyyy',
   'handle': 'example-two.bsky.social',
   ...},
  ...],                     # 50 profiles on this page
 'cursor': '3lx4l7dz2iz26'}  # pass it back to get the next 50
```

Each item is a `profileView`: fewer fields than `get_profile` returns. There are no follower or post counts here; call `get_profile` on a handle when you need them.